### Hyperparameter Tuning using Optuna

In [1]:
# Import necessary libraries
import optuna
import numpy as np
import pandas as pd
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

c:\Users\LOQ\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


#### Load the Pima Indian Diabetes dataset from sklearn.
#### Note: Scikit-learn's built-in 'load_diabetes' is a regression dataset.
#### We will load the actual diabetes dataset from an external source

In [2]:
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
columns = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome']

# Load the dataset
df = pd.read_csv(url, names=columns)
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [3]:
# Replace zero values with NaN in columns where zero is not a valid value
cols_with_missing_vals = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
df[cols_with_missing_vals] = df[cols_with_missing_vals].replace(0, np.nan)

# Input the missing values with the mean of the respective column
df.fillna(df.mean(), inplace=True)

# Check if there are any remaining missing values
print(df.isnull().sum())

Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64


In [4]:
# Split data into features
X = df.drop('Outcome', axis=1)
y = df['Outcome']

# Split data into train, test -> split (70% train, 30% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Optional: Scale the data for better model performance
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Check the shape of the data
print(f'Training set shape: {X_train.shape}')
print(f'Test set shape: {X_test.shape}')

Training set shape: (537, 8)
Test set shape: (231, 8)


In [5]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

# Define the objective function
def objective(trial):
    # Suggest values for the hyperparameters
    n_estimators = trial.suggest_int('n_estimators', 50, 200)
    max_depth = trial.suggest_int('max_depth', 3, 20)

    # Create RandomForestClassifier with suggested hyperparameters
    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=42
    )

    # Perform 3-fold cross-validation and calculate accuracy
    score = cross_val_score(model, X_train, y_train, cv=3, scoring='accuracy').mean()

    return score # Return the accuracy score for Optuna to maximize

In [6]:
# Create a study object and optimize the objective function
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler()) # We aim to maximize accuracy
study.optimize(objective, n_trials=50) # Run 50 trials to find best hyperparameters

[I 2026-08-19 15:56:58,366] A new study created in memory with name: no-name-280cc0a5-3545-40be-8719-0fcc111aa850
[I 2026-08-19 15:56:59,911] Trial 0 finished with value: 0.7690875232774674 and parameters: {'n_estimators': 159, 'max_depth': 17}. Best is trial 0 with value: 0.7690875232774674.
[I 2026-08-19 15:57:00,813] Trial 1 finished with value: 0.7653631284916201 and parameters: {'n_estimators': 120, 'max_depth': 9}. Best is trial 0 with value: 0.7690875232774674.
[I 2026-08-19 15:57:02,906] Trial 2 finished with value: 0.7746741154562384 and parameters: {'n_estimators': 197, 'max_depth': 15}. Best is trial 2 with value: 0.7746741154562384.
[I 2026-08-19 15:57:03,998] Trial 3 finished with value: 0.7653631284916201 and parameters: {'n_estimators': 107, 'max_depth': 6}. Best is trial 2 with value: 0.7746741154562384.
[I 2026-08-19 15:57:05,578] Trial 4 finished with value: 0.7616387337057727 and parameters: {'n_estimators': 146, 'max_depth': 9}. Best is trial 2 with value: 0.7746741

In [7]:
# Print the best result
print(f'Best trial accuracy: {study.best_trial.value}')
print(f'Best hyperparameters: {study.best_trial.params}')

Best trial accuracy: 0.7802607076350093
Best hyperparameters: {'n_estimators': 197, 'max_depth': 16}


In [8]:
from sklearn.metrics import accuracy_score
# Train a RandomForestClassifier using the best hyperparameters from Optuna
best_model = RandomForestClassifier(**study.best_trial.params, random_state=42)

# Fit the model to the training data
best_model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = best_model.predict(X_test)

# Calculate the accuracy on the test set
test_accuracy = accuracy_score(y_test, y_pred)

# Print the test accuracy
print(f'Test Accuracy with best hyperparameters: {test_accuracy:.2f}')


Test Accuracy with best hyperparameters: 0.74


### Samplers in Optuna

In [12]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

# Define the objective function
def objective(trial):
    # Suggest values for the hyperparameters
    n_estimators = trial.suggest_int('n_estimators', 50, 200)
    max_depth = trial.suggest_int('max_depth', 3, 20)

    # Create the RandomForestClassifier with suggested hyperparameters
    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=42
    )

    # Perform 3-fold cross-validation and calculate accuracy
    score = cross_val_score(model, X_train, y_train, cv=3, scoring='accuracy').mean()

    return score  # Return the accuracy score for Optuna to maximize

In [13]:
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.RandomSampler())  # We aim to maximize accuracy
study.optimize(objective, n_trials=50)  # Run 50 trials to find the best hyperparameters

[I 2026-08-19 16:08:54,169] A new study created in memory with name: no-name-04a91b85-0601-4884-a800-bd797a1e4e63
[I 2026-08-19 16:08:54,975] Trial 0 finished with value: 0.7560521415270017 and parameters: {'n_estimators': 103, 'max_depth': 3}. Best is trial 0 with value: 0.7560521415270017.
[I 2026-08-19 16:08:55,830] Trial 1 finished with value: 0.7728119180633147 and parameters: {'n_estimators': 103, 'max_depth': 19}. Best is trial 1 with value: 0.7728119180633147.
[I 2026-08-19 16:08:56,408] Trial 2 finished with value: 0.7746741154562384 and parameters: {'n_estimators': 71, 'max_depth': 16}. Best is trial 2 with value: 0.7746741154562384.
[I 2026-08-19 16:08:57,147] Trial 3 finished with value: 0.7709497206703911 and parameters: {'n_estimators': 92, 'max_depth': 5}. Best is trial 2 with value: 0.7746741154562384.
[I 2026-08-19 16:08:57,667] Trial 4 finished with value: 0.7597765363128491 and parameters: {'n_estimators': 64, 'max_depth': 12}. Best is trial 2 with value: 0.774674115

In [14]:
# Print the best result
print(f'Best trial accuracy: {study.best_trial.value}')
print(f'Best hyperparameters: {study.best_trial.params}')

Best trial accuracy: 0.7802607076350093
Best hyperparameters: {'n_estimators': 118, 'max_depth': 16}


In [15]:
from sklearn.metrics import accuracy_score

# Train a RandomForestClassifier using the best hyperparameters from Optuna
best_model = RandomForestClassifier(**study.best_trial.params, random_state=42)

# Fit the model to the training data
best_model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = best_model.predict(X_test)

# Calculate the accuracy on the test set
test_accuracy = accuracy_score(y_test, y_pred)

# Print the test accuracy
print(f'Test Accuracy with best hyperparameters: {test_accuracy:.2f}')

Test Accuracy with best hyperparameters: 0.74


In [16]:
search_space = {
    'n_estimators': [50, 100, 150, 200],
    'max_depth' : [5, 10, 15, 20]
}

In [11]:
# Create a study and optimize it using GridSampler
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.GridSampler(search_space))
study.optimize(objective)

[I 2026-08-19 16:05:11,392] A new study created in memory with name: no-name-4d032b32-d87d-4ab1-883e-4b1576680aaf
[I 2026-08-19 16:05:12,251] Trial 0 finished with value: 0.7690875232774674 and parameters: {'n_estimators': 100, 'max_depth': 5}. Best is trial 0 with value: 0.7690875232774674.
[I 2026-08-19 16:05:13,380] Trial 1 finished with value: 0.7672253258845437 and parameters: {'n_estimators': 150, 'max_depth': 10}. Best is trial 0 with value: 0.7690875232774674.
[I 2026-08-19 16:05:13,765] Trial 2 finished with value: 0.7728119180633147 and parameters: {'n_estimators': 50, 'max_depth': 15}. Best is trial 2 with value: 0.7728119180633147.
[I 2026-08-19 16:05:14,514] Trial 3 finished with value: 0.7653631284916201 and parameters: {'n_estimators': 100, 'max_depth': 15}. Best is trial 2 with value: 0.7728119180633147.
[I 2026-08-19 16:05:15,289] Trial 4 finished with value: 0.7690875232774674 and parameters: {'n_estimators': 100, 'max_depth': 20}. Best is trial 2 with value: 0.772811

In [17]:

# Print the best result
print(f'Best trial accuracy: {study.best_trial.value}')
print(f'Best hyperparameters: {study.best_trial.params}')

Best trial accuracy: 0.7802607076350093
Best hyperparameters: {'n_estimators': 118, 'max_depth': 16}


In [18]:
from sklearn.metrics import accuracy_score

# Train a RandomForestClassifier using the best hyperparameters from Optuna
best_model = RandomForestClassifier(**study.best_trial.params, random_state=42)

# Fit the model to the training data
best_model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = best_model.predict(X_test)

# Calculate the accuracy on the test set
test_accuracy = accuracy_score(y_test, y_pred)

# Print the test accuracy
print(f'Test Accuracy with best hyperparameters: {test_accuracy:.2f}')

Test Accuracy with best hyperparameters: 0.74


### Optuna Visualizations

In [19]:
# For visualizations
from optuna.visualization import plot_optimization_history, plot_parallel_coordinate, plot_slice, plot_contour, plot_param_importances

# 1. Optimization History
plot_optimization_history(study).show()

In [20]:
# 2. Parallel Coordinates Plot
plot_parallel_coordinate(study).show()

In [21]:
# 3. Slice Plot
plot_slice(study).show()

In [23]:
# 4. Contour Plot
plot_contour(study).show()

In [24]:
# 5. Hyperparameter Importance
plot_param_importances(study).show()

### Optimize Multiple ML Models

In [25]:
# Importing the required libraries
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC

In [26]:
# Define the objective function for Optuna
def objective(trial):
    # Choose the algorithm to tune
    classifier_name = trial.suggest_categorical('classifier', ['SVM', 'RandomForest', 'GradientBoosting'])

    if classifier_name == 'SVM':
        # SVM hyperparameters
        c = trial.suggest_float('C', 0.1, 100, log=True)
        kernel = trial.suggest_categorical('kernel', ['linear', 'rbf', 'poly', 'sigmoid'])
        gamma = trial.suggest_categorical('gamma', ['scale', 'auto'])

        model = SVC(C=c, kernel=kernel, gamma=gamma, random_state=42)

    elif classifier_name == 'RandomForest':
        # Random Forest hyperparameters
        n_estimators = trial.suggest_int('n_estimators', 50, 300)
        max_depth = trial.suggest_int('max_depth', 3, 20)
        min_samples_split = trial.suggest_int('min_samples_split', 2, 10)
        min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 10)
        bootstrap = trial.suggest_categorical('bootstrap', [True, False])

        model = RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            bootstrap=bootstrap,
            random_state=42
        )

    elif classifier_name == 'GradientBoosting':
        # Gradient Boosting hyperparameters
        n_estimators = trial.suggest_int('n_estimators', 50, 300)
        learning_rate = trial.suggest_float('learning_rate', 0.01, 0.3, log=True)
        max_depth = trial.suggest_int('max_depth', 3, 20)
        min_samples_split = trial.suggest_int('min_samples_split', 2, 10)
        min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 10)

        model = GradientBoostingClassifier(
            n_estimators=n_estimators,
            learning_rate=learning_rate,
            max_depth=max_depth,
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            random_state=42
        )

    # Perform cross-validation and return the mean accuracy
    score = cross_val_score(model, X_train, y_train, cv=3, scoring='accuracy').mean()
    return score

In [27]:
# Create a study and optimize it using CmaEsSampler
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=100)

[I 2026-08-19 16:51:47,750] A new study created in memory with name: no-name-f62f3118-e160-4894-80ab-f90a5c8b7a73
[I 2026-08-19 16:51:48,773] Trial 0 finished with value: 0.7430167597765364 and parameters: {'classifier': 'GradientBoosting', 'n_estimators': 261, 'learning_rate': 0.09064294754215058, 'max_depth': 5, 'min_samples_split': 8, 'min_samples_leaf': 10}. Best is trial 0 with value: 0.7430167597765364.
[I 2026-08-19 16:51:49,069] Trial 1 finished with value: 0.7616387337057727 and parameters: {'classifier': 'GradientBoosting', 'n_estimators': 56, 'learning_rate': 0.06946333245978126, 'max_depth': 10, 'min_samples_split': 8, 'min_samples_leaf': 10}. Best is trial 1 with value: 0.7616387337057727.
[I 2026-08-19 16:51:49,094] Trial 2 finished with value: 0.7281191806331471 and parameters: {'classifier': 'SVM', 'C': 1.2340646417505863, 'kernel': 'poly', 'gamma': 'scale'}. Best is trial 1 with value: 0.7616387337057727.
[I 2026-08-19 16:51:50,344] Trial 3 finished with value: 0.73370

In [28]:
# Retrieve best trial
best_trial = study.best_trial
print("Best trial parameters:", best_trial.params)
print("Best trial accuracy:", best_trial.value)

Best trial parameters: {'classifier': 'SVM', 'C': 0.11434321309439659, 'kernel': 'linear', 'gamma': 'auto'}
Best trial accuracy: 0.7895716945996275


In [29]:
study.trials_dataframe()

,number,value,datetime_start,datetime_complete,duration,params_C,params_bootstrap,params_classifier,params_gamma,params_kernel,params_learning_rate,params_max_depth,params_min_samples_leaf,params_min_samples_split,params_n_estimators,state
0,0,0.743017,2026-08-19 16:51:47.752824,2026-08-19 16:51:48.773802,0 days 00:00:01.020978,NaN,NaN,GradientBoosting,NaN,NaN,0.090643,5.0,10.0,8.0,261.0,COMPLETE
1,1,0.761639,2026-08-19 16:51:48.774535,2026-08-19 16:51:49.069513,0 days 00:00:00.294978,NaN,NaN,GradientBoosting,NaN,NaN,0.069463,10.0,10.0,8.0,56.0,COMPLETE
2,2,0.728119,2026-08-19 16:51:49.070268,2026-08-19 16:51:49.094429,0 days 00:00:00.024161,1.234065,NaN,SVM,scale,poly,NaN,NaN,NaN,NaN,NaN,COMPLETE
3,3,0.733706,2026-08-19 16:51:49.095227,2026-08-19 16:51:50.344609,0 days 00:00:01.249382,NaN,NaN,GradientBoosting,NaN,NaN,0.025836,15.0,2.0,3.0,101.0,COMPLETE
4,4,0.761639,2026-08-19 16:51:50.345258,2026-08-19 16:51:51.196097,0 days 00:00:00.850839,NaN,True,RandomForest,NaN,NaN,NaN,14.0,8.0,5.0,237.0,COMPLETE
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,95,0.789572,2026-08-19 16:52:13.567582,2026-08-19 16:52:13.588580,0 days 00:00:00.020998,0.114343,NaN,SVM,auto,linear,NaN,NaN,NaN,NaN,NaN,COMPLETE
96,96,0.787709,2026-08-19 16:52:13.589402,2026-08-19 16:52:13.610595,0 days 00:00:00.021193,0.108248,NaN,SVM,auto,linear,NaN,NaN,NaN,NaN,NaN,COMPLETE
97,97,0.789572,2026-08-19 16:52:13.611348,2026-08-19 16:52:13.631705,0 days 00:00:00.020357,0.131829,NaN,SVM,scale,linear,NaN,NaN,NaN,NaN,NaN,COMPLETE
98,98,0.787709,2026-08-19 16:52:13.632564,2026-08-19 16:52:13.652878,0 days 00:00:00.020314,0.106367,NaN,SVM,scale,linear,NaN,NaN,NaN,NaN,NaN,COMPLETE


In [30]:
study.trials_dataframe()['params_classifier'].value_counts()

params_classifier
SVM                 74
RandomForest        15
GradientBoosting    11
Name: count, dtype: int64

In [32]:
study.trials_dataframe().groupby('params_classifier')['value'].mean()

params_classifier
GradientBoosting    0.744202
RandomForest        0.765984
SVM                 0.774674
Name: value, dtype: float64

In [34]:
# Optimization History
plot_optimization_history(study).show()

In [35]:
# Slice Plot
plot_slice(study).show()

In [36]:
# Hyperparameter Importance
plot_param_importances(study).show()

In [38]:
import optuna
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.datasets import load_iris
from sklearn.metrics import accuracy_score
import numpy as np

# Load the Iris dataset
X, y = load_iris(return_X_y=True)

# Split the dataset into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Define the objective function for XGBoost
def objective(trial):
    # Hyperparameter search space
    param = {
        'verbosity': 0,
        'objective': 'multi:softprob',
        'num_class': 3,
        'eval_metric': 'mlogloss',  # Ensure that the eval_metric is specified here
        'booster': 'gbtree',
        'lambda': trial.suggest_float('lambda', 1e-8, 1.0, log=True),
        'alpha': trial.suggest_float('alpha', 1e-8, 1.0, log=True),
        'eta': trial.suggest_float('eta', 0.01, 0.3),
        'gamma': trial.suggest_float('gamma', 1e-8, 1.0, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 9),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'subsample': trial.suggest_float('subsample', 0.4, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.4, 1.0),
        'n_estimators': 300,
    }

    # Create DMatrix for XGBoost
    dtrain = xgb.DMatrix(X_train, label=y_train)
    dtest = xgb.DMatrix(X_test, label=y_test)

    # Define a pruning callback based on evaluation metrics
    pruning_callback = optuna.integration.XGBoostPruningCallback(trial, "eval-mlogloss")  # Match the metric name in the evals list

    # Train the model
    bst = xgb.train(
        param,
        dtrain,
        num_boost_round=300,
        evals=[(dtrain, "train"), (dtest, "eval")],  # Ensure the eval datasets and names are specified
        early_stopping_rounds=30,
        callbacks=[pruning_callback]
    )

    # Predict on the test set
    preds = bst.predict(dtest)
    best_preds = [int(np.argmax(line)) for line in preds]

    # Return accuracy as the objective value
    accuracy = accuracy_score(y_test, best_preds)
    return accuracy

# Create a study with pruning
study = optuna.create_study(direction='maximize', pruner=optuna.pruners.SuccessiveHalvingPruner())
study.optimize(objective, n_trials=50)

# Output the best trial
print(f"Best trial: {study.best_trial.params}")
print(f"Best accuracy: {study.best_value}")

[I 2026-08-19 16:57:33,895] A new study created in memory with name: no-name-4c622a12-ed47-4a0f-97b8-7350b1525bcc


[0]	train-mlogloss:0.82589	eval-mlogloss:0.81242
[1]	train-mlogloss:0.64488	eval-mlogloss:0.62879
[2]	train-mlogloss:0.51520	eval-mlogloss:0.48789
[3]	train-mlogloss:0.42262	eval-mlogloss:0.39107
[4]	train-mlogloss:0.34923	eval-mlogloss:0.31431
[5]	train-mlogloss:0.29416	eval-mlogloss:0.25632
[6]	train-mlogloss:0.25027	eval-mlogloss:0.20701
[7]	train-mlogloss:0.22797	eval-mlogloss:0.18424
[8]	train-mlogloss:0.19996	eval-mlogloss:0.15674
[9]	train-mlogloss:0.17965	eval-mlogloss:0.13135
[10]	train-mlogloss:0.16801	eval-mlogloss:0.11718
[11]	train-mlogloss:0.15751	eval-mlogloss:0.10397
[12]	train-mlogloss:0.15016	eval-mlogloss:0.09524
[13]	train-mlogloss:0.14655	eval-mlogloss:0.09111
[14]	train-mlogloss:0.14597	eval-mlogloss:0.09078
[15]	train-mlogloss:0.14205	eval-mlogloss:0.08547
[16]	train-mlogloss:0.13861	eval-mlogloss:0.08013
[17]	train-mlogloss:0.13658	eval-mlogloss:0.07741
[18]	train-mlogloss:0.13548	eval-mlogloss:0.07616
[19]	train-mlogloss:0.13500	eval-mlogloss:0.07496
[20]	train

c:\Users\LOQ\anaconda3\Lib\site-packages\optuna\integration\xgboost.py:14: FutureWarning:

`optuna.integration.xgboost` has been deprecated in v4.9.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v4.9.0. Use `optuna_integration.xgboost` instead.

[I 2026-08-19 16:57:34,089] Trial 0 finished with value: 1.0 and parameters: {'lambda': 0.00026044898401805317, 'alpha': 2.0029303880783146e-06, 'eta': 0.22585484883249554, 'gamma': 0.22457657025748454, 'max_depth': 7, 'min_child_weight': 5, 'subsample': 0.8081496973474069, 'colsample_bytree': 0.7378646224969284}. Best is trial 0 with value: 1.0.


[0]	train-mlogloss:1.02428	eval-mlogloss:1.02794
[1]	train-mlogloss:0.94349	eval-mlogloss:0.94230
[2]	train-mlogloss:0.91426	eval-mlogloss:0.91832
[3]	train-mlogloss:0.88427	eval-mlogloss:0.88704
[4]	train-mlogloss:0.83232	eval-mlogloss:0.83360
[5]	train-mlogloss:0.80528	eval-mlogloss:0.80719
[6]	train-mlogloss:0.76078	eval-mlogloss:0.76076
[7]	train-mlogloss:0.73303	eval-mlogloss:0.73148
[8]	train-mlogloss:0.70097	eval-mlogloss:0.70143
[9]	train-mlogloss:0.67334	eval-mlogloss:0.67306
[10]	train-mlogloss:0.66814	eval-mlogloss:0.66967
[11]	train-mlogloss:0.63402	eval-mlogloss:0.63371
[12]	train-mlogloss:0.61705	eval-mlogloss:0.61574
[13]	train-mlogloss:0.59391	eval-mlogloss:0.58942
[14]	train-mlogloss:0.57892	eval-mlogloss:0.57305
[15]	train-mlogloss:0.56249	eval-mlogloss:0.55505
[16]	train-mlogloss:0.54419	eval-mlogloss:0.53409
[17]	train-mlogloss:0.53566	eval-mlogloss:0.52550
[18]	train-mlogloss:0.52224	eval-mlogloss:0.51043
[19]	train-mlogloss:0.51047	eval-mlogloss:0.49696
[20]	train

[I 2026-08-19 16:57:34,641] Trial 1 finished with value: 1.0 and parameters: {'lambda': 3.4993664438965766e-06, 'alpha': 1.047111469771371e-06, 'eta': 0.07371986462813385, 'gamma': 0.4369276370349716, 'max_depth': 3, 'min_child_weight': 9, 'subsample': 0.5896881361720264, 'colsample_bytree': 0.40371561781864607}. Best is trial 0 with value: 1.0.


...

[0]	train-mlogloss:1.03150	eval-mlogloss:1.03163


[I 2026-08-19 16:57:37,528] Trial 48 pruned. Trial was pruned at iteration 1.


[0]	train-mlogloss:0.97982	eval-mlogloss:0.97548


[I 2026-08-19 16:57:37,551] Trial 49 pruned. Trial was pruned at iteration 1.


Best trial: {'lambda': 0.00026044898401805317, 'alpha': 2.0029303880783146e-06, 'eta': 0.22585484883249554, 'gamma': 0.22457657025748454, 'max_depth': 7, 'min_child_weight': 5, 'subsample': 0.8081496973474069, 'colsample_bytree': 0.7378646224969284}
Best accuracy: 1.0


In [39]:
from optuna.visualization import plot_intermediate_values
# Plot intermediate values during the trials
plot_intermediate_values(study).show()